In [0]:
from pyspark.sql import functions as F
from pyspark import pipelines as dp

In [0]:

CATALOG = spark.conf.get("catalog")
BRONZE_SCHEMA = spark.conf.get("bronze_schema")
SILVER_SCHEMA = spark.conf.get("silver_schema")

CLEANED_TABLE = f"{BRONZE_SCHEMA}.cleaned_netflix"
RATING_TABLE = f"{BRONZE_SCHEMA}.rating_reference"
QUARANTINE_TABLE = f"{SILVER_SCHEMA}.quarantine_netflix"

BRONZE_TABLE = f"{BRONZE_SCHEMA}.bronze_netflix"
SILVER_TABLE = f"{SILVER_SCHEMA}.silver_netflix"

In [0]:
EXPECTATIONS = {
    "valid_show_id": "show_id IS NOT NULL",
    "valid_title": "title IS NOT NULL",
    "valid_type": "type IN ('Movie', 'TV Show')",
    "valid_release_year": "release_year BETWEEN 1900 AND 2100"
}

In [0]:
def add_failed_expectations(df):
    failed = []

    for name, condition in EXPECTATIONS.items():
        failed.append(
            F.when(~F.expr(condition), F.lit(name))
        )
    return df.withColumn(
        "failed_expectations", F.array_compact(F.array(*failed))
    )

In [0]:
def clean_netflix(df):
    return (
        df
            .withColumn("show_id", F.trim("show_id"))
            .withColumn(
                "type",
                F.when(F.lower(F.trim(F.col("type"))) == "movie", "Movie")
                .when(F.lower(F.trim(F.col("type"))) == "tv show", "TV Show")
                .otherwise(F.trim(F.col("type")))
            )
            .withColumn("title", F.trim("title"))
            .withColumn("director", F.trim("director"))
            .withColumn("country", F.trim("country"))
            .withColumn("rating", F.upper(F.trim("rating")))
            .withColumn("date_added",  F.try_to_date(F.trim("date_added"), "MMMM d, yyyy"))
            .withColumn("release_year", F.col("release_year").cast("int"))
            .withColumn("duration", F.trim("duration"))
            .withColumn("listed_in", F.trim("listed_in"))
            .withColumn("description", F.trim("description"))
    )

In [0]:
@dp.table(name=CLEANED_TABLE)
def cleaned_netflix():
    df = spark.readStream.table(BRONZE_TABLE)
    return clean_netflix(df)

In [0]:
def enrich_netflix(df):
    return(
        df
        .withColumn(
            "release_period",
            F.when(F.col("release_year") < 2000, "Before 2000")
            .when(F.col("release_year") < 2010, "2000-2009")
            .when(F.col("release_year") < 2020, "2010-2019")
            .otherwise("2020+")
        )
        .withColumn("has_director", F.col("director").isNotNull())
        .withColumn("silver_created_at", F.current_timestamp())
    )

In [0]:
@dp.temporary_view(name="silver_prepared")
@dp.expect_all(EXPECTATIONS)
def silver_netflix():
    df = spark.readStream.table(CLEANED_TABLE)
    reference_df = spark.read.table(RATING_TABLE)
    checked_df = add_failed_expectations(df)
    valid_df = checked_df.filter(F.size("failed_expectations") == 0).drop("failed_expectations")
    enriched = enrich_netflix(valid_df)
    return (
        enriched.join(reference_df, on = "rating", how = "left")
    )

In [0]:
dp.create_streaming_table(name=SILVER_TABLE)
dp.create_auto_cdc_flow(
    target=SILVER_TABLE,
    source="silver_prepared",
    keys=["show_id"],
    sequence_by=F.col("ingestion_time"),
    stored_as_scd_type="2"
)

In [0]:
@dp.table(name=f"{SILVER_SCHEMA}.quarantine_netflix")
def quarantine_netflix():
    df = spark.readStream.table(CLEANED_TABLE)
    checked_df = add_failed_expectations(df)

    return checked_df.filter(
        F.size("failed_expectations") > 0
    )